## Start Ollama

Colab needs `zstd` before the current Ollama installer can unpack successfully. The previous notebook failed here and therefore reached the framework with no Ollama server.

In [23]:
!apt-get update -qq
!apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve >/tmp/ollama.log 2>&1 &
!sleep 5
!ollama --version
!curl -sf http://127.0.0.1:11434/api/tags || (cat /tmp/ollama.log; exit 1)

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
ollama version is 0.33.3
{"models":[{"name":"qwen2.5-coder:7b","model":"qwen2.5-coder:7b","modified_at":"2026-09-04T08:38:24.68982646Z","size":4683087561,"digest":"dae161e27b0e90dd1856c8bb3209201fd6736d8eb66298e75ed87571486f4364","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"7.6B","quantization_level":"Q4_

In [24]:
# Pull the local coding model used by the prototype.
!ollama pull qwen2.5-coder:7b

# SPS Self-Specialization — Minimal Research Prototype

This notebook demonstrates the exact concept: **State 0 (statically programmed IntegerMultiplication) → runtime request → self-replication → Ollama-guided specialization → verification → State 1 (FloatMultiplication)**.

The important behavior is that an unsupported float request does **not** simply fail. The framework uses the existing integer capability as the parent, creates a child, asks the external Ollama model to specialize it, verifies the generated capability, and activates it only after verification succeeds.

In [25]:
# Get the latest repository directly.
%cd /content
!rm -rf self-specialization
!git clone -q https://github.com/muhammadnaumantahir/self-specialization.git
%cd /content/self-specialization
!pip -q install -r requirements.txt pytest

/content
/content/self-specialization


In [26]:
# Deterministic tests: replication, specialization, verification and dispatch.
!PYTHONPATH=. pytest -q

...........                                                              [100%]
11 passed in 0.34s


In [27]:
# Run the actual end-to-end experiment.
import os
os.environ['OLLAMA_MODEL'] = 'qwen2.5-coder:7b'
!PYTHONPATH=. python experiments/self_specialization_demo.py

STATE 0 — STATIC INTEGER MULTIPLICATION
Capability: IntegerMultiplication
Contract: ['int', 'int'] -> int
6 * 7 = 42

USER INPUT — INTEGER
Request: multiply(8, 9)
Resolved capability: IntegerMultiplication
State: S0
Result: 72
Ollama is not needed because State 0 already supports int × int.

USER INPUT — FLOAT (MISSING CAPABILITY)
Request: multiply(2.5, 4.0)
No float capability exists, so the framework will:
  1. detect the unsupported [float, float] contract
  2. replicate IntegerMultiplication
  3. ask Ollama to specialize the child
  4. verify the generated code
  5. integrate and activate FloatMultiplication as State S1
Traceback (most recent call last):
  File "/content/self-specialization/experiments/self_specialization_demo.py", line 109, in <module>
    raise SystemExit(main())
                     ~~~~^^
  File "/content/self-specialization/experiments/self_specialization_demo.py", line 74, in main
    value, specialized = dispatcher.execute(
                         ~~~~~~~~~

## Expected research result

The successful run should show:

```text
STATE 0 — STATIC INTEGER MULTIPLICATION
6 * 7 = 42

RUNTIME REQUEST — INTEGER
8 * 9 = 72
Ollama is not needed because State 0 already supports int × int.

RUNTIME REQUEST — FLOAT (MISSING CAPABILITY)
... replicate → Ollama specialize → verify → activate ...
Result capability: FloatMultiplication
Result state: S1
2.5 * 4.0 = 10.0

SUCCESS: State 0 reproduced and specialized into State 1.
```

The lineage should be `IntegerMultiplication → IntegerMultiplication-child → FloatMultiplication`.

In [ ]:
!PYTHONPATH=. python experiments/self_specialization_demo.py

STATE 0 — STATIC INTEGER MULTIPLICATION
Capability: IntegerMultiplication
Contract: ['int', 'int'] -> int
6 * 7 = 42

USER INPUT — INTEGER
Request: multiply(8, 9)
Resolved capability: IntegerMultiplication
State: S0
Result: 72
Ollama is not needed because State 0 already supports int × int.

USER INPUT — FLOAT (MISSING CAPABILITY)
Request: multiply(2.5, 4.0)
No float capability exists, so the framework will:
  1. detect the unsupported [float, float] contract
  2. replicate IntegerMultiplication
  3. ask Ollama to specialize the child
  4. verify the generated code
  5. integrate and activate FloatMultiplication as State S1
